In [1]:
import pandas as pd

In [2]:
df1 = pd.read_csv('./SampleData.csv',encoding="ISO-8859-1")
df2 = pd.read_csv('./Leads.csv',encoding="ISO-8859-1")

/tmp/ipykernel_70353/3100832126.py:1: DtypeWarning: Columns (0: Website, 1: Date, 2: Next Follow Up, 3: ecode, 4: What attracted you to consider SomeSchool ) have mixed types. Specify dtype option on import or set low_memory=False.
  df1 = pd.read_csv('./SampleData.csv',encoding="ISO-8859-1")


In [3]:
import pandas as pd
import logging

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)
logger = logging.getLogger(__name__)


class DataProcessor:
    """
    A class to handle merging, filtering, and cleaning of two CSV datasets based on lead information.
    """

    def __init__(self, df1_path: str, df2_path: str):
        """
        Initializes the DataProcessor by loading two CSV files into DataFrames.
        """
        try:
            self.df1 = pd.read_csv(df1_path)
            logger.info(f"✅ Loaded first CSV: {df1_path} | Shape: {self.df1.shape}")

            self.df2 = pd.read_csv(df2_path, encoding="ISO-8859-1")
            logger.info(f"✅ Loaded second CSV: {df2_path} | Shape: {self.df2.shape}")

        except FileNotFoundError as e:
            logger.error(f"❌ File not found: {e}")
            raise
        except pd.errors.ParserError as e:
            logger.error(f"❌ Error parsing CSV: {e}")
            raise
        except Exception as e:
            logger.error(f"❌ Unexpected error during initialization: {e}")
            raise

    def _merge_dataframes(self, merge_on_column: str) -> pd.DataFrame:
        """
        Merges two DataFrames on a specified column, avoiding duplication on common columns.
        """
        try:
            common_columns = set(self.df1.columns).intersection(set(self.df2.columns))
            common_columns.remove(merge_on_column)
            logger.debug(f"Common columns (excluding key): {common_columns}")

            merged_df = pd.merge(
                self.df1,
                self.df2.drop(columns=list(common_columns)),
                on=merge_on_column,
                how="outer"
            )

            logger.info(
                f"🔗 Merged DataFrames on '{merge_on_column}' | Shape before: "
                f"{self.df1.shape}, {self.df2.shape} | After merge: {merged_df.shape}"
            )

            return merged_df
        except KeyError as e:
            logger.error(f"❌ Column not found for merging: {e}")
            raise
        except Exception as e:
            logger.error(f"❌ Error while merging DataFrames: {e}")
            raise

    def _drop_columns_with_missing_threshold(
        self, df: pd.DataFrame, threshold: float = 0.6
    ) -> pd.DataFrame:
        """
        Drops columns if missing/empty/null values exceed a given threshold.
        """
        try:
            logger.info(f"🧹 Checking for columns with missing ratio > {threshold:.2f}")

            # Calculate missing ratios
            missing_ratio = df.isnull().mean()
            empty_ratio = (df.astype(str).apply(lambda x: x.str.strip() == "").mean())
            total_missing_ratio = (missing_ratio + empty_ratio) / 2

            cols_to_drop = total_missing_ratio[total_missing_ratio > threshold].index
            logger.debug(f"Missing ratio per column:\n{total_missing_ratio}")

            if len(cols_to_drop) > 0:
                logger.info(
                    f"🚮 Dropping {len(cols_to_drop)} columns exceeding {threshold*100:.0f}% missing threshold."
                )
                logger.info(f"Columns dropped: {list(cols_to_drop)}")

            cleaned_df = df.drop(columns=cols_to_drop)
            logger.info(f"✅ Shape after dropping columns: {cleaned_df.shape}")

            return cleaned_df
        except Exception as e:
            logger.error(f"❌ Error while dropping columns: {e}")
            raise

    def get_filter_data(self, threshold: float = 0.45) -> pd.DataFrame:
        """
        Merges, filters, and cleans the DataFrame to include only relevant leads and remove
        columns with excessive missing values.
        """
        try:
            merged_df = self._merge_dataframes(merge_on_column="Lead Number")

            logger.info(
                f"📊 Starting filter operation | Shape before filtering: {merged_df.shape}"
            )

            # Filter rows
            filtered_df = merged_df[merged_df["Lead Origin"] == "Landing Page Submission"]
            filtered_df = filtered_df.dropna(subset=["Company"])

            logger.info(
                f"✨ Filtered DataFrame | Rows before: {merged_df.shape[0]}, after filtering: {filtered_df.shape[0]}"
            )

            # Drop columns with high missing ratio
            cleaned_df = self._drop_columns_with_missing_threshold(filtered_df, threshold)

            logger.info(
                f"🏁 Final cleaned DataFrame | Shape: {cleaned_df.shape} | Columns retained: {len(cleaned_df.columns)}"
            )
            return cleaned_df

        except KeyError as e:
            logger.error(f"❌ Required column missing during filtering: {e}")
            raise
        except Exception as e:
            logger.error(f"❌ Error during data filtering: {e}")
            raise

    def save_filtered_dataframe(
        self, filtered_dataframe: pd.DataFrame, path: str, index: bool = False
    ) -> str:
        """
        Saves a DataFrame to a CSV file.
        """
        try:
            filtered_dataframe.to_csv(path, index=index)
            logger.info(
                f"💾 DataFrame successfully saved | Path: {path} | Shape: {filtered_dataframe.shape}"
            )
            return f"DataFrame successfully saved to {path}"
        except Exception as e:
            logger.error(f"❌ Failed to save DataFrame: {e}")
            raise

In [4]:
processor= DataProcessor(df2_path='./SampleData.csv',df1_path='./Leads.csv')

2026-02-22 21:35:24,408 | INFO | __main__ | ✅ Loaded first CSV: ./Leads.csv | Shape: (9240, 37)
/tmp/ipykernel_70353/3684302033.py:25: DtypeWarning: Columns (0: Website, 1: Date, 2: Next Follow Up, 3: ecode, 4: What attracted you to consider SomeSchool ) have mixed types. Specify dtype option on import or set low_memory=False.
  self.df2 = pd.read_csv(df2_path, encoding="ISO-8859-1")
2026-02-22 21:35:24,552 | INFO | __main__ | ✅ Loaded second CSV: ./SampleData.csv | Shape: (9240, 122)


In [5]:
merge_df1 = processor.get_filter_data()

2026-02-22 21:35:24,580 | INFO | __main__ | 🔗 Merged DataFrames on 'Lead Number' | Shape before: (9240, 37), (9240, 122) | After merge: (9240, 130)
2026-02-22 21:35:24,581 | INFO | __main__ | 📊 Starting filter operation | Shape before filtering: (9240, 130)
2026-02-22 21:35:24,598 | INFO | __main__ | ✨ Filtered DataFrame | Rows before: 9240, after filtering: 83
2026-02-22 21:35:24,599 | INFO | __main__ | 🧹 Checking for columns with missing ratio > 0.45
2026-02-22 21:35:24,644 | INFO | __main__ | 🚮 Dropping 42 columns exceeding 45% missing threshold.
2026-02-22 21:35:24,645 | INFO | __main__ | Columns dropped: ['Mobile Number', 'Website', 'Time Zone', 'Job Title', 'Order Value', 'Address 1', 'Address 2', 'Zip', 'Facebook URL', 'Twitter URL', 'LinkedIn URL', 'Industry', 'Work Area', 'Course Interested', 'Keyword', 'Date', 'Next Follow Up', 'Any other Please specify', 'Chat Group', 'ecode', 'amt', 'eventname', 'Enquiry Type', 'What attracted you to consider SomeSchool ', 'Landing Page', '

In [6]:
merge_df1.columns.to_list()

['Prospect ID',
 'Lead Number',
 'Lead Origin',
 'Lead Source',
 'Do Not Email',
 'Do Not Call',
 'Converted',
 'TotalVisits',
 'Total Time Spent on Website',
 'Page Views Per Visit',
 'Last Activity',
 'Country',
 'Specialization',
 'How did you hear about X Education',
 'What is your current occupation',
 'What matters most to you in choosing a course',
 'Search',
 'Magazine',
 'Newspaper Article',
 'X Education Forums',
 'Newspaper',
 'Digital Advertisement',
 'Through Recommendations',
 'Receive More Updates About Our Courses',
 'Tags',
 'Lead Quality',
 'Update me on Supply Chain Content',
 'Get updates on DM Content',
 'Lead Profile',
 'City',
 'Asymmetrique Activity Index',
 'Asymmetrique Profile Index',
 'Asymmetrique Activity Score',
 'Asymmetrique Profile Score',
 'I agree to pay the amount through cheque',
 'A free copy of Mastering The Interview',
 'Last Notable Activity',
 'Company',
 'Source Medium',
 'Notes',
 'Source Campaign',
 'Source Content',
 'Lead Stage',
 'Lead G

In [19]:
# merge_df1['Lead Source'].to_list()

In [24]:
merge_df1['City'].unique()

<StringArray>
[         'Other Metro Cities',           'Thane & Outskirts',
                      'Select',                      'Mumbai',
 'Other Cities of Maharashtra',                'Other Cities',
                           nan,              'Tier II Cities']
Length: 8, dtype: str

In [9]:
new_df = merge_df1[['Lead Number','Company','Notes','Specialization','Lead Source']]

In [10]:
new_df

,Lead Number,Company,Notes,Specialization,Lead Source
372,582296,shubham.jain1@maxlifeinsurance.com,"hii, I am looking for distance course of MBA w...",Business Administration,Google
491,583069,Graduate from mumbai university,NaN,Finance Management,Google
766,585174,Kumar Metals,Request to give information about various corr...,Select,Organic Search
807,585443,mumbai university,NaN,Operations Management,Google
852,585811,Social Kinnect,NaN,Select,Organic Search
...,...,...,...,...,...
8623,654061,rakshata.nikam@sharekhan.com,Can you please provide MCOM admission details,"Banking, Investment And Insurance",Direct Traffic
8760,655287,vkaradkar@in.imshealth.com,I am working in a MNC. This is a very good com...,Healthcare Management,Direct Traffic
8968,657572,vinoba bhave university,NaN,Human Resource Management,Referral Sites
9124,659357,Yogeshsadarang@yahoo.in,NaN,Hospitality Management,Google


In [11]:
unique_values = {col: new_df[col].unique().tolist() for col in['Specialization','Lead Source'] }

In [12]:
unique_values

{'Specialization': ['Business Administration',
  'Finance Management',
  'Select',
  'Operations Management',
  'Human Resource Management',
  'Marketing Management',
  'Media and Advertising',
  'Banking, Investment And Insurance',
  'E-Business',
  'Supply Chain Management',
  nan,
  'IT Projects Management',
  'Retail Management',
  'International Business',
  'Hospitality Management',
  'Healthcare Management'],
 'Lead Source': ['Google',
  'Organic Search',
  'Direct Traffic',
  'Referral Sites']}

In [13]:
new_df['Company'].unique()

<StringArray>
[                       'shubham.jain1@maxlifeinsurance.com',
                           'Graduate from mumbai university',
                                              'Kumar Metals',
                                         'mumbai university',
                                            'Social Kinnect',
                                                '8828432104',
                         'nagpur university / October batch',
                                                '9634216295',
                                                '9920086721',
                                            'duplicate lead',
                        'Graduation from Solapur university',
                                            'diploma holder',
                                            'Duplicate lead',
                          'mumbai university / night school',
                'Honeywell Automation India Private Limited',
                               'Punjab Technical Univers